# Entrenamiento y comparación de modelos

Este notebook evalúa modelos con validación temporal expansiva. Recarga desde el archivo original las **26 predictoras conservadas en la exploración final**, el resultado del préstamo y la fecha de otorgamiento. La selección conserva variables con menos del 10% de faltantes en el primer entrenamiento (préstamos elegibles hasta 2012). La lista queda fija para los cuatro cortes; 2017 en adelante se reserva para la prueba.

## Validación que usaremos

Cada corte entrena con años anteriores y valida con el año siguiente. La ventana de entrenamiento crece sin incluir años futuros: primero se entrena hasta 2012 y se valida en 2013; luego hasta 2013 y se valida en 2014; hasta 2014 y se valida en 2015; y hasta 2015 y se valida en 2016. Esta evaluación de origen móvil permite observar si el rendimiento se mantiene al avanzar en el tiempo (Tashman, 2000). Sigue el principio de los cortes temporales expansivos descritos por scikit-learn: los conjuntos de entrenamiento aumentan de un corte al siguiente (scikit-learn developers, n.d.-a).

No usamos una partición aleatoria común porque queremos simular el caso práctico de entrenar con préstamos antiguos y predecir préstamos posteriores. La validación aleatoria puede responder otra pregunta —interpolar entre préstamos de años ya vistos—, mientras que estos cortes miden el cambio al pasar a cohortes posteriores. La literatura no prohíbe K-fold para todo problema temporal; su validez depende del problema y de sus supuestos (Bergmeir et al., 2018). Los cuatro años de validación sirven para comparar los modelos y sus configuraciones. No se usarán para entrenar dentro de su propio corte. El conjunto desde 2017 se reserva para una evaluación final después de escoger el enfoque; no se utilizará para decidir qué modelo gana.

En cada corte, la mediana, los indicadores (en el escenario que los incluye), las categorías de one-hot encoding y el escalado se aprenden únicamente con los años de entrenamiento. Comparamos dos alternativas: mediana sola y mediana más indicadores para las numéricas que tengan algún faltante en ese entrenamiento. Luego se aplican a la validación de ese corte. Ajustar esas transformaciones con validación o prueba trasladaría información de esos periodos al entrenamiento y podría inflar las métricas (scikit-learn developers, n.d.-b).

## Cómo trataremos los valores faltantes

La exploración final revisa 95 predictoras candidatas en el primer entrenamiento, formado por préstamos elegibles hasta 2012. Se conservan las 26 que tienen menos del 10% de faltantes. El 10% es un criterio práctico para fijar una lista utilizable desde el primer corte, no un límite universal.

En cada corte, las variables numéricas se rellenan con la mediana calculada en ese entrenamiento y las categóricas con `Sin información`. Comparamos dos versiones de cada modelo: sin indicadores y con un indicador 0/1 para cada variable numérica que tenga faltantes en el entrenamiento. La validación recibe la mediana y las columnas aprendidas solo en su entrenamiento; nunca se calcula una mediana con validación o prueba.

Las tablas imputadas se construyen un corte por vez y se liberan al terminarlo. La función reutilizable está en `src/modelado.py`; aquí se muestra un corte de ejemplo y se comparan ambas versiones de los modelos.


## 1. Cargar los datos conservados en la exploración final

El kernel de este notebook no comparte la RAM del notebook exploratorio, así que vuelve a leer el archivo fuente. La función carga directamente las 26 predictoras que quedaron tras aplicar el criterio de faltantes hasta 2012, junto con la variable objetivo y la fecha para construir los cortes.

In [ ]:
from pathlib import Path
import sys
import time
import gc
import pandas as pd
import numpy as np
from IPython.display import display

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.config import localizar_datos, ANIO_CORTE_PRUEBA
from src.modelado import (
    NOMBRES_VARIABLES,
    cargar_datos_modelado,
    crear_cortes_temporales,
    crear_tablas_imputadas_corte,
    modelos_iniciales,
    evaluar_modelos_temporales,
    resumir_metricas,
)

archivo = localizar_datos()
print("Archivo fuente:", archivo)
inicio = time.perf_counter()
datos, predictoras = cargar_datos_modelado(archivo)
tiempo_carga = time.perf_counter() - inicio
memoria = datos.memory_usage(index=True, deep=True).sum() / 1024**3
print(f"Registros elegibles cargados: {len(datos):,}")
print(f"Predictoras conservadas: {len(predictoras)}")
print(f"Columnas en la tabla de modelado (predictoras + fecha/año/objetivo): {datos.shape[1]}")
print(f"Tiempo de carga y preparación: {tiempo_carga:.1f} segundos")
print(f"Memoria profunda de la tabla: {memoria:.2f} GiB")
print("Criterio aplicado: menos del 10% de faltantes en el primer entrenamiento (hasta 2012).")

distribucion = datos["default"].value_counts().sort_index().rename(
    index={0: "Fully Paid", 1: "Charged Off"}
).to_frame("préstamos")
distribucion["porcentaje"] = distribucion["préstamos"] / len(datos) * 100
display(distribucion)


## 2. Definir los cortes temporales

En cada fila se muestra el año final del entrenamiento y el año usado para validar. Los registros desde 2017 se identifican como prueba reservada; no entran a la comparación durante esta etapa.

In [ ]:
tabla_cortes = crear_cortes_temporales(
    datos,
    primer_anio_validacion=2013,
    ultimo_anio_validacion=2016,
    anio_prueba=ANIO_CORTE_PRUEBA,
)
display(tabla_cortes)


## 3. Ejemplo de las tablas imputadas para un corte

El primer corte usa préstamos emitidos hasta 2012 para entrenar y los de 2013 para validar. La mediana se calcula solo con entrenamiento y se aplica a ambos conjuntos. Comparamos dos matrices que comparten las mismas filas y predictores: una con mediana sola y otra que además conserva indicadores 0/1 de faltante para las variables numéricas que tuvieron faltantes en entrenamiento. Las categóricas se rellenan con `Sin información`.

In [ ]:
corte_ejemplo = crear_tablas_imputadas_corte(
    datos, predictoras, anio_final_entrenamiento=2012,
    anio_validacion=2013, usar_indicadores=True,
)
corte_ejemplo_sin_indicadores = crear_tablas_imputadas_corte(
    datos, predictoras, anio_final_entrenamiento=2012,
    anio_validacion=2013, usar_indicadores=False,
)

print("Con indicadores — X entrenamiento:", corte_ejemplo["X_entrenamiento"].shape)
print("Sin indicadores — X entrenamiento:", corte_ejemplo_sin_indicadores["X_entrenamiento"].shape)
print("Filas idénticas:", corte_ejemplo["X_entrenamiento"].index.equals(corte_ejemplo_sin_indicadores["X_entrenamiento"].index))
print("Indicadores numéricos creados:", len(corte_ejemplo["columnas_indicadoras"]))
print("Faltantes restantes en X entrenamiento:", int(corte_ejemplo["X_entrenamiento"].isna().sum().sum()))
print("Faltantes restantes en X validación:", int(corte_ejemplo["X_validacion"].isna().sum().sum()))

medianas_ejemplo = corte_ejemplo["medianas"].rename(index=lambda c: NOMBRES_VARIABLES.get(c, c)).to_frame(
    "mediana_calculada_hasta_2012"
)
print("Medianas aprendidas usando solo el entrenamiento:")
display(medianas_ejemplo)
print("Primeras filas imputadas del entrenamiento:")
display(corte_ejemplo["X_entrenamiento"].head().rename(columns=lambda c: NOMBRES_VARIABLES.get(c, c)))
print("Primeras filas imputadas de validación:")
display(corte_ejemplo["X_validacion"].head().rename(columns=lambda c: NOMBRES_VARIABLES.get(c, c)))

# No guardamos las tablas completas del ejemplo; la función volverá a generarlas al evaluar.
del corte_ejemplo, corte_ejemplo_sin_indicadores, medianas_ejemplo
gc.collect()


## 4. Modelos a comparar

La función admite un diccionario de estimadores de scikit-learn, por lo que se pueden agregar modelos o cambiar sus parámetros desde este notebook. Empezaremos con un clasificador mayoritario como referencia, regresión logística, árbol de decisión y Gradient Boosting. Estas configuraciones son puntos de partida, no resultados de una búsqueda exhaustiva de hiperparámetros.

La métrica principal será ROC-AUC y la secundaria Average Precision (área bajo la curva precisión-recall). También reportaremos KS, recall, precisión, F1, exactitud y tiempo de entrenamiento por corte. Para estas métricas con umbral usamos 0.5 de manera fija; la comparación principal se basa en ROC-AUC y Average Precision, que no dependen de ese umbral.

In [ ]:
modelos = modelos_iniciales()
print("Modelos que se evaluarán:")
for nombre, modelo in modelos.items():
    print(f"- {nombre}: {modelo}")


## 5. Comparar modelos con y sin indicadores

Ejecutamos los mismos modelos y cortes dos veces: primero con la imputación por mediana sin indicadores y luego con un indicador para cada predictor numérico que tenga faltantes en el entrenamiento de ese corte. Los predictores, filas, particiones temporales, algoritmo e hiperparámetros se mantienen iguales. Las medianas y el codificador se ajustan de nuevo dentro de cada entrenamiento. La prueba queda reservada.

In [ ]:
inicio_modelos = time.perf_counter()
resultados_variantes = []
tablas_medianas = []
for usar_indicadores in [False, True]:
    nombre_variante = "mediana + indicadores" if usar_indicadores else "mediana sola"
    print(f"Evaluando {nombre_variante}.", flush=True)
    resultado_variante, medianas_variante = evaluar_modelos_temporales(
        datos, predictoras, modelos=modelos,
        primer_anio_validacion=2013, ultimo_anio_validacion=2016,
        usar_indicadores=usar_indicadores,
        umbral_clasificacion=0.5, verbose=True,
    )
    resultados_variantes.append(resultado_variante)
    if not usar_indicadores:
        tablas_medianas = medianas_variante
    else:
        pd.testing.assert_frame_equal(tablas_medianas, medianas_variante)
    del resultado_variante, medianas_variante
    gc.collect()
resultados_por_corte = pd.concat(resultados_variantes, ignore_index=True)
tiempo_modelos = time.perf_counter() - inicio_modelos

columnas_metricas = [
    "modelo", "indicadores_faltantes", "corte", "n", "tasa_default_real",
    "roc_auc", "pr_auc", "ks", "recall", "precision", "f1", "accuracy",
    "segundos_entrenamiento",
]
print(f"Tiempo total de entrenamiento y evaluación: {tiempo_modelos / 60:.1f} minutos")
print("Resultados por modelo, variante de faltantes y año:")
display(resultados_por_corte[columnas_metricas].round(4))

resumen_modelos = resumir_metricas(resultados_por_corte)
print("Promedio y variación entre los cuatro años de validación:")
display(resumen_modelos.round(4))

print("Medianas utilizadas por cada corte de entrenamiento:")
tablas_medianas.index = [NOMBRES_VARIABLES.get(c, c) for c in tablas_medianas.index]
display(tablas_medianas.round(2))

## 6. Guardar resultados y dejar la prueba reservada

Los valores bajos de recall y F1 en algunos cortes se deben leer junto al umbral fijo usado aquí (0.5 para probabilidades y 0 para el margen SVM). Esta primera comparación prioriza la capacidad de ordenar los préstamos, medida con ROC-AUC y Average Precision; más adelante el umbral se puede elegir con una validación interna del entrenamiento, sin optimizarlo sobre el mismo año que se reporta.

Las advertencias de categorías desconocidas indican que apareció en validación alguna categoría que no estaba en el entrenamiento de ese corte. `OneHotEncoder(handle_unknown="ignore")` la representa con ceros; no se pierden filas ni se detiene el ajuste.

Guardamos las métricas por corte con la columna `indicadores_faltantes`, que separa mediana sola de mediana más indicadores, y el resumen por modelo y variante. El conjunto 2017 en adelante no se usó para construir estas métricas. Después de decidir el modelo y sus parámetros con validación, se volverá a ajustar con todo el periodo de desarrollo anterior a 2017 y se hará una sola evaluación final sobre la prueba. Antes de interpretar esa evaluación, hay que revisar la maduración de los préstamos de 2017–2018: al conservar solo resultados finales, las cohortes recientes pueden estar representadas de forma distinta.


In [ ]:
salida = RAIZ / "outputs" / "entrenamiento_26_variables"
salida.mkdir(parents=True, exist_ok=True)
resultados_por_corte.to_csv(salida / "metricas_validacion_temporal.csv", index=False, encoding="utf-8-sig")
resumen_modelos.to_csv(salida / "resumen_modelos_validacion.csv", encoding="utf-8-sig")
tablas_medianas.to_csv(salida / "medianas_por_corte.csv", encoding="utf-8-sig")
print("Resultados guardados en:", salida)


## Resultados guardados de la referencia con 26 variables

Esta tabla corresponde a la ejecución de los cuatro cortes temporales con las configuraciones iniciales. Los archivos completos por corte están en `outputs/entrenamiento_26_variables/`.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
carpeta_resultados = RAIZ / "outputs" / "entrenamiento_26_variables"
resumen_guardado = pd.read_csv(carpeta_resultados / "resumen_modelos_validacion.csv", index_col=0)
metricas_guardadas = pd.read_csv(carpeta_resultados / "metricas_validacion_temporal.csv", encoding="utf-8-sig")
display(resumen_guardado.round(4))
display(metricas_guardadas[["modelo", "indicadores_faltantes", "anio_validacion", "roc_auc", "pr_auc", "segundos_entrenamiento"]].round(4))

,indicadores_faltantes,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,ks_mean,ks_std,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std,accuracy_mean,accuracy_std,segundos_entrenamiento_mean,segundos_entrenamiento_std
modelo,,,,,,,,,,,,,,,,,
gradient_boosting,False,0.6954,0.0171,0.3509,0.0631,0.2812,0.0260,0.0087,0.0120,0.4170,0.3021,0.0169,0.0231,0.8070,0.0310,6.5455,3.7810
gradient_boosting,True,0.6953,0.0172,0.3508,0.0635,0.2810,0.0260,0.0086,0.0121,0.4066,0.3088,0.0167,0.0232,0.8070,0.0310,6.0822,4.7089
regresion_logistica,True,0.6926,0.0183,0.3482,0.0647,0.2792,0.0256,0.0189,0.0145,0.5464,0.1122,0.0362,0.0270,0.8074,0.0303,29.3438,25.1030
regresion_logistica,False,0.6925,0.0184,0.3481,0.0649,0.2790,0.0258,0.0192,0.0142,0.5461,0.1153,0.0367,0.0265,0.8073,0.0302,24.3072,20.2253
svm_lineal,False,0.6916,0.0193,0.3464,0.0657,0.2777,0.0270,0.0015,0.0024,0.4734,0.1935,0.0030,0.0048,0.8064,0.0320,9.2731,8.8600
svm_lineal,True,0.6916,0.0194,0.3463,0.0655,0.2775,0.0274,0.0015,0.0025,0.4568,0.1875,0.0030,0.0050,0.8063,0.0320,11.5496,10.0295
arbol_decision,True,0.6709,0.0173,0.3201,0.0587,0.2466,0.0228,0.0041,0.0082,0.1533,0.3065,0.0080,0.0159,0.8066,0.0316,4.3660,4.0447
arbol_decision,False,0.6709,0.0173,0.3201,0.0587,0.2466,0.0228,0.0041,0.0082,0.1533,0.3065,0.0080,0.0159,0.8066,0.0316,4.4810,3.9740
baseline_mayoria,False,0.5000,0.0000,0.1938,0.0322,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8062,0.0322,0.0184,0.0132


,modelo,indicadores_faltantes,anio_validacion,roc_auc,pr_auc,segundos_entrenamiento
0,baseline_mayoria,False,2013,0.5000,0.1560,0.0034
1,regresion_logistica,False,2013,0.6690,0.2618,7.8765
2,svm_lineal,False,2013,0.6673,0.2583,1.8153
3,arbol_decision,False,2013,0.6533,0.2433,0.8296
4,gradient_boosting,False,2013,0.6746,0.2667,3.9649
5,baseline_mayoria,False,2014,0.5000,0.1845,0.0187
6,regresion_logistica,False,2014,0.6948,0.3346,11.5225
7,svm_lineal,False,2014,0.6953,0.3341,4.2730
8,arbol_decision,False,2014,0.6744,0.3054,2.2351
9,gradient_boosting,False,2014,0.6976,0.3384,3.6125


## Referencias

Bergmeir, C., Hyndman, R. J., & Koo, B. (2018). A note on the validity of cross-validation for evaluating autoregressive time series prediction. *Computational Statistics & Data Analysis, 120*, 70–83. https://doi.org/10.1016/j.csda.2017.11.003

Madley-Dowd, P., Hughes, R. A., Tilling, K., & Heron, J. (2019). The proportion of missing data should not be used to guide decisions on multiple imputation. *Journal of Clinical Epidemiology, 110*, 63–73. https://doi.org/10.1016/j.jclinepi.2019.02.016

Perez-Lebel, A., Varoquaux, G., Le Morvan, M., Josse, J., & Poline, J.-B. (2022). Benchmarking missing-values approaches for predictive models on health databases. *GigaScience, 11*, giac013. https://doi.org/10.1093/gigascience/giac013

Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting, 16*(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0

scikit-learn developers. (n.d.-a). *TimeSeriesSplit*. scikit-learn User Guide. Retrieved November 9, 2026, from https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html

scikit-learn developers. (n.d.-b). *Common pitfalls and recommended practices: Data leakage*. scikit-learn User Guide. Retrieved November 9, 2026, from https://scikit-learn.org/stable/common_pitfalls.html
